# 03 · Diagnose the RGB baseline

Inspect saved checkpoints using the training split: class-conditional probabilities, predicted water fractions and gradient statistics. This diagnostic does not apply optimizer updates.

In the original run, RGB selected epoch 1 and RGB+NIR selected epoch 36. The RGB checkpoint assigned similar mean water probabilities to water and non-water pixels (0.1875 and 0.1900). Its behavior motivated the fixed-crop fit in Notebook 04.

## 1. Select a saved run

Set `RUN_NAME` to the experiment of interest. An empty value selects the latest compatible directory with both checkpoints. Check the printed path and saved epochs. Results are written under that run's `diagnostics/` directory.

In [ ]:
from pathlib import Path
import csv
import json
import numpy as np
import rasterio
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import Dataset

ROOT = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
RUN_NAME = ""
completed = sorted(p for p in (ROOT / "runs").glob("rgb_nir_*")
                   if (p / "rgb/best.pt").exists() and (p / "rgb_nir/best.pt").exists())
RUN = ROOT / "runs" / RUN_NAME if RUN_NAME else completed[-1]
OUT = RUN / "diagnostics"
OUT.mkdir(exist_ok=True)
device = torch.device("cuda")
print("Inspecting:", RUN)
print("GPU:", torch.cuda.get_device_name(0))
BASE = ROOT / "data/sen1floods11/v1.1"
HAND = BASE / "data/flood_events/HandLabeled"
with (BASE / "splits/flood_handlabeled/flood_train_data.csv").open(newline="") as f:
    train_rows = list(csv.reader(f))
CFG = json.loads((RUN / "config.json").read_text(encoding="utf-8"))["training"]

## 2. Reconstruct preprocessing and model

Use the earlier 64/128/256/512/1024 U-Net, the shared four-band validity mask and full 512 × 512 training images. Reflectance is DN/10000. If the checkpoint configuration includes channel means and standard deviations, reuse them.

Probability analysis uses FP32. The next gradient diagnostic compares FP32 with AMP separately.

In [ ]:
class FloodDataset(Dataset):
    def __init__(self, rows, channels, training=False):
        self.rows = rows
        self.channels = channels
        self.training = training
        self.epoch = 0

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, index):
        image_name, label_name = self.rows[index]
        image_name = image_name.replace("_S1Hand", "_S2Hand")
        with rasterio.open(HAND / "S2Hand" / image_name) as src:
            bands = [src.descriptions.index(b) + 1 for b in ["B4", "B3", "B2", "B8"]]
            x = src.read(bands, masked=True).astype("float32").filled(np.nan)
        with rasterio.open(HAND / "LabelHand" / label_name) as src:
            y = src.read(1).astype("int64")
        y[~np.isfinite(x).all(axis=0)] = -1
        x = x / 10000.0
        if "channel_mean" in CFG:
            mean = np.asarray(CFG["channel_mean"], dtype=np.float32)
            std = np.asarray(CFG["channel_std"], dtype=np.float32)
            x = (x - mean[:, None, None]) / std[:, None, None]
        x = np.nan_to_num(x)
        if self.training:
            rng = np.random.default_rng([CFG["seed"], self.epoch, index])
            size = CFG["crop_size"]
            top = int(rng.integers(0, y.shape[0] - size + 1))
            left = int(rng.integers(0, y.shape[1] - size + 1))
            x = x[:, top:top+size, left:left+size]
            y = y[top:top+size, left:left+size]
            for axis in [0, 1]:
                if rng.random() < 0.5:
                    x = np.flip(x, axis=axis + 1)
                    y = np.flip(y, axis=axis)
        return torch.from_numpy(x[:self.channels].copy()), torch.from_numpy(y.copy())


import torch
from torch import nn

class DoubleConv(nn.Sequential):
    def __init__(self, in_channels, out_channels):
        super().__init__(
            nn.Conv2d(in_channels, out_channels, 3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, 3, padding=1),
            nn.ReLU(inplace=True),
        )

class UNet(nn.Module):
    def __init__(self, in_channels=3, num_classes=2):
        super().__init__()
        self.encoders = nn.ModuleList([
            DoubleConv(in_channels, 64), DoubleConv(64, 128),
            DoubleConv(128, 256), DoubleConv(256, 512),
        ])
        self.pool = nn.MaxPool2d(2)
        self.bottleneck = DoubleConv(512, 1024)
        self.upsamples = nn.ModuleList([
            nn.ConvTranspose2d(1024, 512, 2, stride=2),
            nn.ConvTranspose2d(512, 256, 2, stride=2),
            nn.ConvTranspose2d(256, 128, 2, stride=2),
            nn.ConvTranspose2d(128, 64, 2, stride=2),
        ])
        self.decoders = nn.ModuleList([
            DoubleConv(1024, 512), DoubleConv(512, 256),
            DoubleConv(256, 128), DoubleConv(128, 64),
        ])
        self.head = nn.Conv2d(64, num_classes, 1)

    def forward(self, x):
        skips = []
        for encoder in self.encoders:
            x = encoder(x)
            skips.append(x)
            x = self.pool(x)
        x = self.bottleneck(x)
        for upsample, decoder, skip in zip(self.upsamples, self.decoders, reversed(skips)):
            x = upsample(x)
            x = decoder(torch.cat([skip, x], dim=1))
        return self.head(x)



def load_model(name):
    checkpoint = torch.load(RUN / name / "best.pt", map_location="cpu", weights_only=True)
    model = UNet(in_channels=len(checkpoint["bands"]), num_classes=2)
    model.load_state_dict(checkpoint["model"])
    print(name, "saved epoch:", checkpoint["epoch"], "bands:", checkpoint["bands"])
    return model.to(device).eval(), checkpoint["epoch"]

def save_csv(path, rows):
    with path.open("w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=list(rows[0]))
        writer.writeheader()
        writer.writerows(rows)

def show_table(rows):
    columns = list(rows[0])
    lines = ["| " + " | ".join(columns) + " |", "|" + "---|" * len(columns)]
    for row in rows:
        lines.append("| " + " | ".join(f"{row[k]:.6g}" if isinstance(row[k], float) else str(row[k]) for k in columns) + " |")
    display(Markdown("\n".join(lines)))

## 3. Probability distributions on the training split

Aggregate all 252 training images. Report class prevalence, predicted water fraction, cross-entropy, water IoU and mean water probability within each true class. Histograms use 50 bins over 0–1, normalized separately for water and non-water.

Overlapping distributions reveal poor class separation. The saved checkpoint is evaluated with its original argmax rule.

In [ ]:
dataset = FloodDataset(train_rows, channels=4)
summary, histograms = [], {}
bins = np.linspace(0, 1, 51)
for name, channels in [("rgb", 3), ("rgb_nir", 4)]:
    model, epoch = load_model(name)
    counts = np.zeros(2, dtype=np.int64)
    probability_sums = np.zeros(2, dtype=np.float64)
    squares = 0.0
    loss_sum = 0.0
    matrix = np.zeros((2, 2), dtype=np.int64)
    hist = np.zeros((2, 50), dtype=np.int64)
    for i in range(len(dataset)):
        x, y = dataset[i]
        valid = y != -1
        if not valid.any():
            continue
        with torch.inference_mode():
            logits = model(x[:channels].unsqueeze(0).to(device))
            loss_sum += F.cross_entropy(logits, y.unsqueeze(0).to(device),
                                       ignore_index=-1, reduction="sum").item()
            probability = logits.softmax(1)[0, 1].cpu().numpy()
            pred = logits.argmax(1)[0].cpu().numpy()
        truth = y.numpy()
        keep = valid.numpy()
        matrix += np.bincount(2 * truth[keep] + pred[keep], minlength=4).reshape(2, 2)
        for cls in [0, 1]:
            values = probability[keep & (truth == cls)].astype(np.float64)
            counts[cls] += len(values)
            probability_sums[cls] += values.sum()
            squares += np.square(values).sum()
            hist[cls] += np.histogram(values, bins=bins)[0]
        if (i + 1) % 50 == 0:
            print(name, i + 1, "/", len(dataset), flush=True)
    total = int(counts.sum())
    p = counts[1] / total
    mean = probability_sums.sum() / total
    tn, fp, fn, tp = matrix.ravel()
    summary.append(dict(experiment=name, saved_epoch=epoch,
        true_water_fraction=float(p), predicted_water_fraction=float((tp + fp) / total),
        probability_on_water=float(probability_sums[1] / counts[1]),
        probability_on_nonwater=float(probability_sums[0] / counts[0]),
        probability_std=float(np.sqrt(max(0, squares / total - mean**2))),
        train_cross_entropy=float(loss_sum / total),
        constant_probability_cross_entropy=float(-p*np.log(p)-(1-p)*np.log(1-p)),
        train_global_water_iou=float(tp / (tp + fp + fn))))
    histograms[name] = hist
    del model
save_csv(OUT / "train_probability_summary.csv", summary)
show_table(summary)
fig, axes = plt.subplots(1, 2, figsize=(11, 4), layout="constrained")
centres = (bins[:-1] + bins[1:]) / 2
for ax, (name, hist) in zip(axes, histograms.items()):
    for cls, label in [(0, "True non-water"), (1, "True water")]:
        ax.plot(centres, hist[cls] / hist[cls].sum(), label=label)
    ax.axvline(0.5, color="gray", linestyle="--", label="Decision boundary")
    ax.set(title=name, xlabel="Predicted water probability", ylabel="Fraction within true class", xlim=(0, 1))
    ax.legend()
fig.savefig(OUT / "train_probability_histogram.png", dpi=150)
plt.show()
plt.close(fig)

## 4. Gradient check on one crop

Use the center 256 × 256 crop of training image `India_285297`. Compare first-layer, bottleneck and output-layer gradient norms under FP32 and AMP, and report the decoder's zero-activation fraction. AMP gradients are unscaled before comparison. No optimizer step is taken.

Finite gradients establish that backpropagation works for this checkpoint and crop; they do not reconstruct unsaved training epochs.

In [ ]:
sample_index = next(i for i, row in enumerate(train_rows) if row[0] == "India_285297_S1Hand.tif")
x, y = dataset[sample_index]
x = x[:3, 128:384, 128:384].unsqueeze(0).to(device)
y = y[128:384, 128:384].unsqueeze(0).to(device)
print("Sample: India_285297, valid pixels:", int((y != -1).sum()), "water pixels:", int((y == 1).sum()))
model, epoch = load_model("rgb")
features = {}
def capture_features(module, inputs, output):
    features["zero_fraction"] = float((output.detach() == 0).float().mean())
handle = model.decoders[-1].register_forward_hook(capture_features)
gradient_rows = []
parameters = dict(model.named_parameters())
for use_amp in [False, True]:
    model.zero_grad(set_to_none=True)
    with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
        logits = model(x)
        loss = F.cross_entropy(logits, y, ignore_index=-1)
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
    scaler.scale(loss).backward()
    scale = scaler.get_scale()
    for layer in ["encoders.0.0.weight", "bottleneck.0.weight", "head.weight", "head.bias"]:
        grad = parameters[layer].grad.detach().float() / scale
        gradient_rows.append(dict(precision="AMP" if use_amp else "FP32", layer=layer,
            loss=float(loss.detach()), gradient_l2=float(grad.norm()),
            gradient_finite=bool(torch.isfinite(grad).all()),
            decoder_zero_fraction=features["zero_fraction"]))
handle.remove()
model.zero_grad(set_to_none=True)
del model
save_csv(OUT / "rgb_gradient_check.csv", gradient_rows)
show_table(gradient_rows)
print("Diagnostic output:", OUT)

## Outcome

The returned diagnostic showed little RGB class separation and finite, nonzero FP32/AMP gradients on the checked crop. Notebook 04 therefore tested whether a fresh RGB model could fit that fixed crop.